# 🤖 CEM4644 · MP5 — One model for everything? A vision-language model for classification, detection and take-off
## Workshop (in class): *Façade defects, site safety, floor plans*

**No coding needed.** Each grey box below is one *step*: click the ▶ (play) button at its left, wait until it finishes, look at the result, then answer the report question that follows. Run the steps **from top to bottom**.

In MP2, MP3 and MP4 you trained or used one **specialist** model per task: a classifier, a detector, a segmentation model. This lab gives all three tasks to one **generalist**: a large multimodal language model (Gemini), which has never seen our photos and is steered only by the words you send it. The lab is built around two questions: *can a prompt replace a trained model?* and *how do you get an answer a program can read?*

**What you will do (about 90 minutes)**
1. Talk to the model about a photo, and get the answer as JSON in two ways: by asking nicely, and by enforcing a schema.
2. Classify the MP2 photos with three prompts and compare with the MP2 model.
3. Detect and count on the MP3 photos, compare with the MP3 model.
4. Measure rooms on the MP4 plans: the model's own polygons, or its boxes handed to SAM 3.
5. Your own image and your own words in a small app.

**Before you start**
- **Gemini API key (free).** Open https://aistudio.google.com/apikey, sign in with your Google account, create a key. In Colab, click the **key icon** in the left bar (*Secrets*), add a secret named `GEMINI_API_KEY` with the key as its value, and switch on *Notebook access*. Without a key the precomputed answers of the built-in examples still work; your own prompts and images do not.
- **GPU (optional):** *Runtime → Change runtime type → T4 GPU* is only needed for Step 4b (SAM 3). Everything else runs on a remote service and needs no GPU.
- Never paste your key into a cell you might share: use the secret.

In [ ]:
#@title ▶ Step 0 · Run me first (1–3 minutes) { display-mode: "form" }
#@markdown Click ▶ and wait for the green ✅ line. This downloads the examples with their precomputed answers, and (if *load_sam* is ticked) SAM 3 for Step 4b (about 3 GB).
#@markdown Leave *api_key* empty to use the Colab secret GEMINI_API_KEY (recommended). Untick *load_sam* if you have no GPU and want to skip Step 4b's live part.
api_key = "" #@param {type:"string"}
model = "gemini-3.5-flash-lite" #@param ["gemini-3.5-flash-lite", "gemini-3.5-flash", "gemini-3.8-flash", "gemini-3.6-flash"]
load_sam = True #@param {type:"boolean"}
import importlib, os, shutil, subprocess, sys
REPO, FOLDER, PKG = "CEM4644", "mp5_llm_vision", "aec_llm"

def _git(*args):
    return subprocess.run(["git", "-C", REPO, *args], capture_output=True, text=True).returncode == 0

if os.path.isdir(REPO):                      # a copy is already here: pull the newest course code over it
    if not (_git("fetch", "-q", "--depth", "1", "origin", "master")
            and _git("reset", "-q", "--hard", "FETCH_HEAD") and _git("clean", "-qfd")):
        shutil.rmtree(REPO, ignore_errors=True)          # broken copy: start again from scratch
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "https://github.com/Haolan-Zhang/CEM4644.git", REPO], check=True)
for _m in [m for m in list(sys.modules) if m.split(".")[0] in (PKG, "aec_seg")]:
    del sys.modules[_m]                      # Python caches imported code: drop it, or this cell keeps the old version
importlib.invalidate_caches()
sys.path.insert(0, os.path.abspath(os.path.join(REPO, FOLDER)))
from aec_llm import lab
lab.setup(dataset="workshop", api_key=api_key, model=model, load_sam=load_sam)


## Part 1 · Talk to the model

A **vision-language model** reads an image and text together and answers in text. It has no fixed list of classes and no output layer for boxes: whatever structure you want back, you must **ask for it in words**, and the reply is a piece of text that a program then has to read. That is the whole difference from the specialists: the prompt is the program.

Two things to watch in every step: the **reply itself** (is it what you asked for, is it right?) and its **cost**: seconds per request and **tokens** (the units the service bills; an image costs a few hundred tokens, the model's private *thinking* costs more).

In [ ]:
#@title ▶ Step 1a · The examples and their answer keys { display-mode: "form" }
#@markdown The same kind of material as in MP2, MP3 and MP4, with the answer keys and with what the earlier labs' specialist models said about them.
which = "photos" #@param ["photos", "site photos", "plans"]
lab.show_examples(which)


In [ ]:
#@title ▶ Step 1b · Ask anything about a photo { display-mode: "form" }
#@markdown Free text in, free text out. The three questions below are precomputed for the first photo; any other photo or question runs live (needs your key).
photo = "plain_1  (truth: plain wall (no defect))" #@param ["plain_1  (truth: plain wall (no defect))", "plain_2  (truth: plain wall (no defect))", "minor_crack_1  (truth: minor crack)", "minor_crack_2  (truth: minor crack)", "major_crack_1  (truth: major crack)", "major_crack_2  (truth: major crack)", "spalling_1  (truth: spalling)", "spalling_2  (truth: spalling)", "peeling_1  (truth: peeling paint / plaster)", "peeling_2  (truth: peeling paint / plaster)", "stain_1  (truth: stain)", "stain_2  (truth: stain)", "algae_1  (truth: algae / biological growth)", "algae_2  (truth: algae / biological growth)"]
question = "What do you see in this photo? Answer in three sentences." #@param ["What do you see in this photo? Answer in three sentences.", "Is there anything a building inspector should worry about here? Answer in two sentences.", "Describe this photo as a JSON object with the keys \"what\", \"condition\" and \"action\"."] {allow-input: true}
lab.describe(photo, question)


In [ ]:
#@title ▶ Step 1c · Getting JSON, two ways { display-mode: "form" }
#@markdown **A:** the prompt asks for JSON, nothing enforces it. **B:** the same prompt with a **JSON schema** handed to the API, which rejects any reply that does not fit. Each way is run several times: watch whether the format and the label stay the same.
photo = "plain_1  (truth: plain wall (no defect))" #@param ["plain_1  (truth: plain wall (no defect))", "plain_2  (truth: plain wall (no defect))", "minor_crack_1  (truth: minor crack)", "minor_crack_2  (truth: minor crack)", "major_crack_1  (truth: major crack)", "major_crack_2  (truth: major crack)", "spalling_1  (truth: spalling)", "spalling_2  (truth: spalling)", "peeling_1  (truth: peeling paint / plaster)", "peeling_2  (truth: peeling paint / plaster)", "stain_1  (truth: stain)", "stain_2  (truth: stain)", "algae_1  (truth: algae / biological growth)", "algae_2  (truth: algae / biological growth)"]
repeats = 3 #@param {type:"slider", min:1, max:3, step:1}
lab.json_lab(photo, repeats)


> ### 📝 Report question 1
> From Step 1c: how many of the plain replies (A) were valid JSON, and did the label stay the same across the runs? What did the schema (B) change, and what did it not change? Why does a program that has to read the reply (to fill a table, to count, to draw a box) need B rather than A?

## Part 2 · Classification by prompt

The MP2 task again: building façade / wall surface defects, 7 classes, and a model that was never trained on them. Three prompts of increasing length are prepared: the class names only, the names with a one-line description each, and the descriptions plus decision rules. The reply is scored against the answer key exactly as in MP2, and the MP2 course model's accuracy on the same photos is shown next to it.

In [ ]:
#@title ▶ Step 2a · Classify all the photos { display-mode: "form" }
#@markdown All three prompts are precomputed with the schema on; *basic* is also precomputed with the schema off. Other combinations run live (a few minutes on the free tier).
prompt = "basic" #@param ["basic", "with descriptions", "with descriptions and rules"]
schema = True #@param {type:"boolean"}
show_mistakes = True #@param {type:"boolean"}
lab.classify(prompt, schema, show_mistakes)


In [ ]:
#@title ▶ Step 2b · Your own prompt { display-mode: "form" }
#@markdown Edit the text (keep `{classes}` where the list of categories should go; `{intro}` is the first sentence). Runs live on every photo: needs your key; the free tier allows only a few requests per minute, so this takes one to three minutes.
prompt_text = "{intro} Classify it into exactly one of these categories: {classes}. Reply with JSON only, no other text, in this form: {\"label\": <one category, spelled exactly as in the list>, \"confidence\": <a number from 0 to 1>, \"reason\": <one short sentence>}" #@param {type:"string"}
schema = True #@param {type:"boolean"}
lab.classify_own(prompt_text, schema)


> ### 📝 Report question 2
> From Step 2a: the accuracy of the three prompts (basic / with descriptions / with descriptions and rules) and of the MP2 model on the same 14 photos. Which classes does Gemini confuse (use the confusion table), and what did the descriptions and the rules change?

> ### 📝 Report question 3
> From Step 2b: what did you change in the prompt and what accuracy did you get? If it went up, what is the risk of tuning a prompt on the same photos you score it on (think of MP2's training / test split)?

## Part 3 · Detection and counting by prompt

The MP3 task: workers and PPE on construction sites. The prompt asks for a list of boxes as `[ymin, xmin, ymax, xmax]` on a 0–1000 grid (the convention this model was trained with), the notebook converts them to pixels and scores them like MP3 did: a box is *found* when its label is right and it overlaps the answer key's box by at least half. The MP3 YOLO model's boxes on the same photos are shown next to Gemini's.

In [ ]:
#@title ▶ Step 3a · Boxes on one photo { display-mode: "form" }
site = "site_1  (11 boxes in the answer key)" #@param ["site_1  (11 boxes in the answer key)", "site_2  (8 boxes in the answer key)", "site_3  (9 boxes in the answer key)", "site_4  (6 boxes in the answer key)", "site_5  (12 boxes in the answer key)", "site_6  (6 boxes in the answer key)"]
schema = True #@param {type:"boolean"}
lab.detect(site, schema)


In [ ]:
#@title ▶ Step 3b · All the photos, scored { display-mode: "form" }
schema = True #@param {type:"boolean"}
lab.detect_all(schema)


In [ ]:
#@title ▶ Step 3c · Just ask for the number { display-mode: "form" }
#@markdown Instead of boxes, the model is asked for a count. Compared with the answer key and with counting the model's own boxes from Step 3a.
site = "site_1  (11 boxes in the answer key)" #@param ["site_1  (11 boxes in the answer key)", "site_2  (8 boxes in the answer key)", "site_3  (9 boxes in the answer key)", "site_4  (6 boxes in the answer key)", "site_5  (12 boxes in the answer key)", "site_6  (6 boxes in the answer key)"]
schema = True #@param {type:"boolean"}
lab.count(site, schema)


> ### 📝 Report question 4
> From Step 3b: Gemini's recall and precision against the MP3 YOLO model's. Which label is hardest for Gemini (helmet, NO helmet, vest, NO vest, person) and why might that be? Paste one overlay from Step 3a and explain the extras (thick boxes marked '?').

> ### 📝 Report question 5
> From Step 3c on two photos: the model's count, the count of its own boxes and the answer key. When they disagree, which one is wrong and how would you know on a site where there is no answer key? Which of the two ways of counting would you trust on a site camera, and why?

## Part 4 · Rooms on a floor plan

The MP4 task on three of the MP4 workshop plans: clean renderings of real Finnish homes with every room's real area in the answer key. Two ways to get square metres out of a language model:

- **LLM only:** the prompt asks for each room's outline as a polygon (a list of points on the 0–1000 grid) and its label; the notebook converts the polygon to pixels and to m² with the plan's scale.
- **LLM boxes + SAM 3:** the prompt asks only for a box per room; each box is handed to SAM 3 exactly as your own boxes were in MP4 (*empty room* + box, holes filled, walls removed). The language model does the *finding and naming*, the segmentation model does the *pixels*.

Both are scored against the drawing's answer key, and MP4's result (SAM 3 asked for *room* by phrase) is shown for comparison.

In [ ]:
#@title ▶ Step 4a · The model's own polygons { display-mode: "form" }
#@markdown Precomputed for every plan with the schema on and off. Try both: without the schema this model tends to skip the polygons and only give boxes, and long lists of coordinates are where the plain JSON breaks.
plan = "1293: flat with four large rooms" #@param ["1293: flat with four large rooms", "2536: flat with three bedrooms", "2090: small flat with a balcony"]
mode = "LLM only (polygons)" #@param ["LLM only (polygons)"]
schema = True #@param {type:"boolean"}
lab.segment(plan, mode, schema)


In [ ]:
#@title ▶ Step 4b · The model's boxes, SAM 3's pixels { display-mode: "form" }
#@markdown The model's boxes are precomputed; SAM 3 runs live on them (GPU: seconds; CPU: about a minute per plan). Without SAM 3 loaded, the box areas are used.
plan = "1293: flat with four large rooms" #@param ["1293: flat with four large rooms", "2536: flat with three bedrooms", "2090: small flat with a balcony"]
mode = "LLM boxes + SAM 3" #@param ["LLM boxes + SAM 3"]
schema = True #@param {type:"boolean"}
lab.segment(plan, mode, schema)


In [ ]:
#@title ▶ Step 4c · Room by room, three ways { display-mode: "form" }
plan = "1293: flat with four large rooms" #@param ["1293: flat with four large rooms", "2536: flat with three bedrooms", "2090: small flat with a balcony"]
lab.segment_compare(plan)


> ### 📝 Report question 6
> From Step 4c on one plan: copy the per-room table. Which way is closer to the drawing, the model's own polygons or its boxes handed to SAM 3, and on which rooms do they differ most? How does this compare with drawing the boxes yourself in MP4?

## Part 5 · Your image, your words

A small app: pick a built-in image or upload your own, choose a task (it fills in a starting prompt), edit the words, switch the schema on or off, and read the raw reply next to what the notebook draws from it. Needs your key: everything here is live.

In [ ]:
#@title ▶ Step 5 · Prompt lab { display-mode: "form" }
#@markdown If the app does not appear, run the cell again; if a public link is printed, it also works on a phone.
lab.prompt_app()


> ### 📝 Report question 7
> Run at least 2 experiments of your own in Step 5 (a photo from a site or from the internet, a plan, a changed prompt, the schema on and off). For each: the image, the prompt, the raw reply, and whether it was right. What kind of request broke the model, and how did it break (wrong answer, invented objects, unreadable reply)?

## Wrap-up · Generalist or specialist?

In [ ]:
#@title ▶ Step 6 · All tasks side by side { display-mode: "form" }
#@markdown One table: the generalist with one prompt per task against the three specialists from MP2, MP3 and MP4, with time and tokens.
lab.summary()


> ### 📝 Report question 8
> From Step 6: for each task, would you use the generalist, the specialist, or both together (as in Step 4b)? Argue with the numbers you got and with what each needs: labelled data, training, a GPU, a network connection, money per request, and someone who checks. What does structured output guarantee about a reply, and what does it not guarantee?

In [ ]:
#@title ▶ Numbers for your report { display-mode: "form" }
lab.report_summary()


### Credits
- Photos: BD3 Building Defect Dataset (CC-BY-4.0), the test photos of MP2.
- Site photos: Roboflow 100 'construction-safety' (CC-BY-4.0), the test photos of MP3.
- Floor plans: CubiCasa5K (CC BY-NC-SA 4.0), prepared for MP4 (plans 1293, 2536, 2090).
- Model: Gemini (Google) through the Gemini API, free tier; SAM 3 (Meta, SAM License) from the MP4 folder.
- Lab code: https://github.com/Haolan-Zhang/CEM4644 (folder `mp5_llm_vision`).